In [ ]:
###############################################################################
# Cell 1: Setup - Quarter dates, day fractions, house data, construction/sales
###############################################################################
import numpy as np
from datetime import date
from scipy.optimize import brentq

# Quarter end dates (16 quarters: Q1 2017 through Q4 2020)
quarter_end_dates = [
    date(2017, 3, 31), date(2017, 6, 30), date(2017, 9, 30), date(2017, 12, 31),
    date(2018, 3, 31), date(2018, 6, 30), date(2018, 9, 30), date(2018, 12, 31),
    date(2019, 3, 31), date(2019, 6, 30), date(2019, 9, 30), date(2019, 12, 31),
    date(2020, 3, 31), date(2020, 6, 30), date(2020, 9, 30), date(2020, 12, 31),
]

quarter_start_dates = [
    date(2017, 1, 1), date(2017, 4, 1), date(2017, 7, 1), date(2017, 10, 1),
    date(2018, 1, 1), date(2018, 4, 1), date(2018, 7, 1), date(2018, 10, 1),
    date(2019, 1, 1), date(2019, 4, 1), date(2019, 7, 1), date(2019, 10, 1),
    date(2020, 1, 1), date(2020, 4, 1), date(2020, 7, 1), date(2020, 10, 1),
]

# Days in each quarter
days_in_quarter = [(quarter_end_dates[i] - quarter_start_dates[i]).days + 1 for i in range(16)]

# Days in calendar year for each quarter (determines leap year handling)
def days_in_year(yr):
    if yr % 4 == 0 and (yr % 100 != 0 or yr % 400 == 0):
        return 366
    return 365

days_in_yr = [days_in_year(quarter_end_dates[i].year) for i in range(16)]

# Quarter fractions for interest/fee calculation: days_in_quarter / days_in_year
qfrac = [days_in_quarter[i] / days_in_yr[i] for i in range(16)]

print('Quarter day counts:', days_in_quarter)
print('Year day counts:', days_in_yr)
print('Quarter fractions:', [round(f, 8) for f in qfrac])

###############################################################################
# House categories: (size_sqm, cost_per_sqm, sales_price)
###############################################################################
house_info = {
    'A': (50, 2600, 250_000),
    'B': (60, 2400, 300_000),
    'C': (75, 2300, 335_000),
    'D': (95, 2500, 500_000),
}

# Construction cost per house = size * cost_per_sqm
cost_per_house = {cat: info[0] * info[1] for cat, info in house_info.items()}
print('Cost per house:', cost_per_house)
# A=130000, B=144000, C=172500, D=237500

# Houses built per quarter (16 entries, quarters 5-12 have builds)
houses_built = {
    'A': [0,0,0,0, 200,100,70,70, 50,50,0,0, 0,0,0,0],
    'B': [0,0,0,0, 0,100,150,100, 80,80,0,0, 0,0,0,0],
    'C': [0,0,0,0, 0,0,0,50, 100,100,50,50, 0,0,0,0],
    'D': [0,0,0,0, 0,0,0,0, 30,50,100,50, 0,0,0,0],
}

# Houses sold per quarter (16 entries, quarters 7-16 have sales)
houses_sold = {
    'A': [0,0,0,0,0,0, 40,160,104,73, 66,52,40,5,0,0],
    'B': [0,0,0,0,0,0, 0,20,100,135, 101,82,64,8,0,0],
    'C': [0,0,0,0,0,0, 0,0,0,10, 55,95,90,55,40,5],
    'D': [0,0,0,0,0,0, 0,0,0,0, 6,31,58,85,45,5],
}

# Verify totals match: A=540, B=510, C=350, D=230 => 1630 total
for cat in 'ABCD':
    b = sum(houses_built[cat])
    s = sum(houses_sold[cat])
    print(f'Cat {cat}: built={b}, sold={s}')
    assert b == s, f'Mismatch for {cat}!'

In [ ]:
###############################################################################
# Cell 2: Operating Cash Flows -> Questions 1 & 2
###############################################################################

# Construction costs per quarter
construction_costs = [0.0] * 16
for q in range(16):
    for cat in 'ABCD':
        construction_costs[q] += houses_built[cat][q] * cost_per_house[cat]

# Sales revenue per quarter
sales_revenue = [0.0] * 16
for q in range(16):
    for cat in 'ABCD':
        sales_revenue[q] += houses_sold[cat][q] * house_info[cat][2]

# Other costs
land_costs      = [80_000_000] + [0]*15
earthworks      = [20_000_000, 17_500_000, 15_000_000, 15_000_000] + [0]*12
pre_construction = [2_500_000, 5_000_000, 7_500_000, 9_000_000] + [0]*12
other_costs     = [2_500_000]*4 + [1_000_000]*8 + [0]*4

total_other = [land_costs[i] + earthworks[i] + pre_construction[i] + other_costs[i] for i in range(16)]

# Operating cash flow = sales revenue - construction costs - other costs
operating_cf = [sales_revenue[i] - construction_costs[i] - total_other[i] for i in range(16)]

print('Construction costs per Q:', [f'${c:,.0f}' for c in construction_costs])
print('Sales revenue per Q:', [f'${r:,.0f}' for r in sales_revenue])
print('Other costs per Q:', [f'${c:,.0f}' for c in total_other])
print('Operating CF per Q:', [f'${c:,.0f}' for c in operating_cf])
print(f'Total operating CF: ${sum(operating_cf):,.0f}')

# QUESTION 1: Construction cost for quarter ended March 2019 (Q9, index 8)
q1_value = construction_costs[8]
print(f'\nQ1: Construction cost Mar 2019 = ${q1_value:,.0f}')

# Match to multiple choice
q1_options = {'A':42394000,'B':42395000,'C':42396000,'D':42397000,'E':42398000,
              'F':42399000,'G':42400000,'H':42401000,'I':42402000}
q1_answer = min(q1_options, key=lambda k: abs(q1_options[k] - q1_value))
print(f'Q1 answer: {q1_answer} (${q1_options[q1_answer]:,})')

# QUESTION 2: Total operating cash flow over the project
q2_value = sum(operating_cf)
print(f'\nQ2: Total operating CF = ${q2_value:,.0f}')

q2_options = {'A':72110000,'B':72111000,'C':72112000,'D':72113000,'E':72114000,
              'F':72115000,'G':72116000,'H':72117000,'I':72118000}
q2_answer = min(q2_options, key=lambda k: abs(q2_options[k] - q2_value))
print(f'Q2 answer: {q2_answer} (${q2_options[q2_answer]:,})')

In [ ]:
###############################################################################
# Cell 3: Debt Models and Iterative Solver -> Questions 3-7
###############################################################################

def run_debt_model(opcf, interest_rate=0.07,
                   fixed_arranging_fee=None,
                   variable_arranging_fee_pct=None,
                   commitment_fee_rate=0.0,
                   equity_schedule='fixed'):
    """
    Run the quarterly debt model with iterative convergence.
    
    Parameters:
    - opcf: list of 16 operating cash flows
    - interest_rate: annual interest rate (default 7%)
    - fixed_arranging_fee: if set, fixed fee paid in Q1
    - variable_arranging_fee_pct: if set, fee = pct * max_facility (iterative)
    - commitment_fee_rate: annual rate on undrawn amount (0 = none)
    - equity_schedule: 'fixed' ($10M/Q for Q1-Q4) or 'variable' (20% of need)
    
    Returns dict with all quarterly arrays and summary stats.
    """
    total_equity_limit = 40_000_000
    max_facility_guess = 0.0
    
    for iteration in range(2000):
        # Determine arranging fee
        if fixed_arranging_fee is not None:
            arranging_fee = fixed_arranging_fee
        elif variable_arranging_fee_pct is not None:
            arranging_fee = variable_arranging_fee_pct * max_facility_guess
        else:
            arranging_fee = 0.0
        
        debt_opening = [0.0] * 16
        debt_closing = [0.0] * 16
        interest = [0.0] * 16
        commit_fees = [0.0] * 16
        debt_draw = [0.0] * 16
        debt_repay = [0.0] * 16
        distributions = [0.0] * 16
        equity_draws = [0.0] * 16
        cumulative_equity = 0.0
        
        for q in range(16):
            if q > 0:
                debt_opening[q] = debt_closing[q - 1]
            
            # Interest on opening balance
            interest[q] = debt_opening[q] * interest_rate * qfrac[q]
            
            # Commitment fee on undrawn amount at start of quarter
            if commitment_fee_rate > 0:
                undrawn = max(0.0, max_facility_guess - debt_opening[q])
                commit_fees[q] = undrawn * commitment_fee_rate * qfrac[q]
            
            # Arranging fee in Q1 only
            arr_fee_q = arranging_fee if q == 0 else 0.0
            
            # Net cash = operating CF - interest - arranging fee - commitment fee
            net_cf = opcf[q] - interest[q] - arr_fee_q - commit_fees[q]
            
            if equity_schedule == 'fixed':
                # Fixed: $10M per quarter for Q1-Q4
                eq_this_q = 10_000_000 if q < 4 else 0.0
                equity_draws[q] = eq_this_q
                total_available = net_cf + eq_this_q
                
                if total_available < 0:
                    debt_draw[q] = -total_available
                    debt_repay[q] = 0.0
                    distributions[q] = 0.0
                else:
                    debt_draw[q] = 0.0
                    repay = min(total_available, debt_opening[q])
                    debt_repay[q] = repay
                    distributions[q] = total_available - repay
            
            elif equity_schedule == 'variable':
                # Alternative: equity = 20% of funding need, until $40M cap
                if net_cf < 0:
                    funding_need = -net_cf
                    remaining_equity = total_equity_limit - cumulative_equity
                    if remaining_equity > 0:
                        eq_this_q = min(0.20 * funding_need, remaining_equity)
                    else:
                        eq_this_q = 0.0
                    equity_draws[q] = eq_this_q
                    cumulative_equity += eq_this_q
                    debt_draw[q] = funding_need - eq_this_q
                    debt_repay[q] = 0.0
                    distributions[q] = 0.0
                else:
                    equity_draws[q] = 0.0
                    debt_draw[q] = 0.0
                    repay = min(net_cf, debt_opening[q])
                    debt_repay[q] = repay
                    distributions[q] = net_cf - repay
            
            debt_closing[q] = debt_opening[q] + debt_draw[q] - debt_repay[q]
        
        new_max = max(debt_closing)
        
        # Check convergence (no iteration needed for fixed arranging fee)
        if fixed_arranging_fee is not None and commitment_fee_rate == 0:
            max_facility_guess = new_max
            break
        
        if abs(new_max - max_facility_guess) < 0.01:
            max_facility_guess = new_max
            break
        max_facility_guess = new_max
    
    return {
        'debt_opening': debt_opening, 'debt_closing': debt_closing,
        'interest': interest, 'commit_fees': commit_fees,
        'debt_draw': debt_draw, 'debt_repay': debt_repay,
        'distributions': distributions, 'equity_draws': equity_draws,
        'max_facility': max_facility_guess,
        'arranging_fee': arranging_fee,
        'iterations': iteration + 1,
    }

# ---------- XIRR function ----------
def xirr(cashflows, dates, guess=0.1):
    """Calculate XIRR (internal rate of return for irregular cash flows)."""
    t0 = dates[0]
    def npv(rate):
        return sum(cf / (1 + rate) ** ((d - t0).days / 365.0)
                   for cf, d in zip(cashflows, dates))
    return brentq(npv, -0.5, 10.0, xtol=1e-12)

# =====================================================================
# SCENARIO 1: Fixed $5M arranging fee, no commitment fee, fixed equity
# (Questions 1-3)
# =====================================================================
s1 = run_debt_model(operating_cf, interest_rate=0.07,
                    fixed_arranging_fee=5_000_000,
                    commitment_fee_rate=0.0,
                    equity_schedule='fixed')

print('=== Scenario 1: Fixed arranging fee, no commitment fee ===')
for q in range(16):
    print(f'Q{q+1:2d}: open={s1["debt_opening"][q]:>16,.2f}  '
          f'int={s1["interest"][q]:>12,.2f}  '
          f'draw={s1["debt_draw"][q]:>14,.2f}  '
          f'repay={s1["debt_repay"][q]:>14,.2f}  '
          f'close={s1["debt_closing"][q]:>16,.2f}  '
          f'dist={s1["distributions"][q]:>14,.2f}')
print(f'Max facility: ${s1["max_facility"]:,.2f}')

# Q3: Debt drawdown in Q ended June 2017 = Q2 (index 1)
q3_value = round(s1['debt_draw'][1])
print(f'\nQ3: Debt draw Q2 = ${q3_value:,}')
q3_options = {'A':16745197,'B':16745198,'C':16745199,'D':16745200,'E':16745201,
              'F':16745202,'G':16745203,'H':16745204,'I':16745205}
q3_answer = min(q3_options, key=lambda k: abs(q3_options[k] - q3_value))
print(f'Q3 answer: {q3_answer}')

# =====================================================================
# SCENARIO 2: Variable arranging fee (2% of max), no commitment fee
# (Question 4)
# =====================================================================
s2 = run_debt_model(operating_cf, interest_rate=0.07,
                    variable_arranging_fee_pct=0.02,
                    commitment_fee_rate=0.0,
                    equity_schedule='fixed')

print(f'\n=== Scenario 2: Variable arranging fee, no commitment fee ===')
print(f'Max facility: ${s2["max_facility"]:,.2f}')
print(f'Arranging fee: ${s2["arranging_fee"]:,.2f}')
print(f'Iterations: {s2["iterations"]}')

q4_value = round(s2['arranging_fee'])
q4_options = {'A':4782284,'B':4782285,'C':4782286,'D':4782287,'E':4782288,
              'F':4782289,'G':4782290,'H':4782291,'I':4782292}
q4_answer = min(q4_options, key=lambda k: abs(q4_options[k] - q4_value))
print(f'Q4: Arranging fee rounded = ${q4_value:,}, answer: {q4_answer}')

# =====================================================================
# SCENARIO 3: Variable arranging fee + commitment fee 1.25%
# (Questions 5, 6, 7)
# =====================================================================
s3 = run_debt_model(operating_cf, interest_rate=0.07,
                    variable_arranging_fee_pct=0.02,
                    commitment_fee_rate=0.0125,
                    equity_schedule='fixed')

print(f'\n=== Scenario 3: Variable arranging fee + commitment fee ===')
print(f'Max facility: ${s3["max_facility"]:,.2f}')
print(f'Arranging fee: ${s3["arranging_fee"]:,.2f}')

q5_value = round(s3['max_facility'])
q5_options = {'A':241794916,'B':241794917,'C':241794918,'D':241794919,'E':241794920,
              'F':241794921,'G':241794922,'H':241794923,'I':241794924}
q5_answer = min(q5_options, key=lambda k: abs(q5_options[k] - q5_value))
print(f'Q5: Max facility rounded = ${q5_value:,}, answer: {q5_answer}')

# Q6: IRR at 7% interest (scenario 3)
eq_cfs_6 = []
eq_dates_6 = []
for q in range(16):
    cf = -s3['equity_draws'][q] + s3['distributions'][q]
    if cf != 0:
        eq_cfs_6.append(cf)
        eq_dates_6.append(quarter_end_dates[q])

irr_7 = xirr(eq_cfs_6, eq_dates_6)
print(f'\nQ6: IRR at 7% = {irr_7*100:.4f}%')

q6_options = {'A':16.6,'B':16.7,'C':16.8,'D':16.9,'E':17.0,
              'F':17.1,'G':17.2,'H':17.3,'I':17.4}
q6_val_rounded = round(irr_7 * 100, 1)
q6_answer = min(q6_options, key=lambda k: abs(q6_options[k] - q6_val_rounded))
print(f'Q6: IRR rounded to 1dp = {q6_val_rounded}%, answer: {q6_answer}')

# Q7: IRR at 6.5% interest (same structure as scenario 3 but 6.5% rate)
s3_65 = run_debt_model(operating_cf, interest_rate=0.065,
                       variable_arranging_fee_pct=0.02,
                       commitment_fee_rate=0.0125,
                       equity_schedule='fixed')

eq_cfs_7 = []
eq_dates_7 = []
for q in range(16):
    cf = -s3_65['equity_draws'][q] + s3_65['distributions'][q]
    if cf != 0:
        eq_cfs_7.append(cf)
        eq_dates_7.append(quarter_end_dates[q])

irr_65 = xirr(eq_cfs_7, eq_dates_7)
print(f'\nQ7: IRR at 6.5% = {irr_65*100:.4f}%')

q7_options = {'A':18.4,'B':18.5,'C':18.6,'D':18.7,'E':18.8,
              'F':18.9,'G':19.0,'H':19.1,'I':19.2}
q7_val_rounded = round(irr_65 * 100, 1)
q7_answer = min(q7_options, key=lambda k: abs(q7_options[k] - q7_val_rounded))
print(f'Q7: IRR rounded to 1dp = {q7_val_rounded}%, answer: {q7_answer}')

In [ ]:
###############################################################################
# Cell 4: Alternative Equity Drawdown -> Questions 8 & 9
###############################################################################

# Q8-Q9 use alternative equity (20% of funding need), with variable arranging
# fee and commitment fee (same as scenario 3 but variable equity)

s4 = run_debt_model(operating_cf, interest_rate=0.07,
                    variable_arranging_fee_pct=0.02,
                    commitment_fee_rate=0.0125,
                    equity_schedule='variable')

print('=== Scenario 4: Alternative equity drawdown ===')
cum_eq = 0.0
for q in range(16):
    cum_eq += s4['equity_draws'][q]
    print(f'Q{q+1:2d}: equity={s4["equity_draws"][q]:>14,.2f}  '
          f'cum_eq={cum_eq:>14,.2f}  '
          f'debt_close={s4["debt_closing"][q]:>16,.2f}  '
          f'dist={s4["distributions"][q]:>14,.2f}')

print(f'\nMax facility: ${s4["max_facility"]:,.2f}')
print(f'Total equity drawn: ${sum(s4["equity_draws"]):,.2f}')

# Q8: Undrawn equity remaining at end of quarter ended December 2017 (Q4, index 3)
total_equity_drawn_q4 = sum(s4['equity_draws'][:4])
undrawn_eq_q4 = 40_000_000 - total_equity_drawn_q4
q8_answer = round(undrawn_eq_q4)
print(f'\nQ8: Equity drawn through Q4 = ${total_equity_drawn_q4:,.2f}')
print(f'Q8: Undrawn equity end Q4 = ${undrawn_eq_q4:,.2f}')
print(f'Q8 answer: {q8_answer}')

# Q9: Total size of debt facility
q9_answer = round(s4['max_facility'])
print(f'\nQ9: Max facility = ${s4["max_facility"]:,.2f}')
print(f'Q9 answer: {q9_answer}')

In [ ]:
###############################################################################
# Cell 5: Question 10 - Land price for 12% IRR
###############################################################################

def compute_irr_for_land_price(land_price):
    """Compute equity IRR under alternative equity drawdown with a given land price."""
    # Recompute operating CF with new land price
    new_land = [land_price] + [0]*15
    new_other = [new_land[i] + earthworks[i] + pre_construction[i] + other_costs[i] for i in range(16)]
    new_opcf = [sales_revenue[i] - construction_costs[i] - new_other[i] for i in range(16)]
    
    # Run model with alternative equity, variable arranging fee, commitment fee
    res = run_debt_model(new_opcf, interest_rate=0.07,
                         variable_arranging_fee_pct=0.02,
                         commitment_fee_rate=0.0125,
                         equity_schedule='variable')
    
    # Build equity cash flows for XIRR
    cfs = []
    dts = []
    for q in range(16):
        cf = -res['equity_draws'][q] + res['distributions'][q]
        if cf != 0:
            cfs.append(cf)
            dts.append(quarter_end_dates[q])
    
    if not cfs:
        return None
    return xirr(cfs, dts)

# Find land price where IRR = 12%
# Higher land price -> more costs -> lower returns -> lower IRR
def irr_minus_target(land_price):
    irr = compute_irr_for_land_price(land_price)
    if irr is None:
        return -1.0
    return irr - 0.12

optimal_land = brentq(irr_minus_target, 80_000_000, 120_000_000, xtol=100)
optimal_land_rounded = round(optimal_land / 100_000) * 100_000

print(f'Q10: Exact land price for 12% IRR = ${optimal_land:,.2f}')
print(f'Q10: Rounded to nearest $100K = ${optimal_land_rounded:,.0f}')
q10_answer = int(optimal_land_rounded)
print(f'Q10 answer: {q10_answer}')

In [ ]:
###############################################################################
# Cell 6: Compile all answers
###############################################################################

answers = {
    'question1':  q1_answer,
    'question2':  q2_answer,
    'question3':  q3_answer,
    'question4':  q4_answer,
    'question5':  q5_answer,
    'question6':  q6_answer,
    'question7':  q7_answer,
    'question8':  q8_answer,
    'question9':  q9_answer,
    'question10': q10_answer,
}

print('=== FINAL ANSWERS ===')
for k, v in sorted(answers.items()):
    print(f'  {k}: {v}')